# WiDS Wildfire Survival Superensemble


In [1]:
import os, re, glob, json, math, random, warnings, gc
from collections import defaultdict
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

SEED = 20260420
random.seed(SEED)
np.random.seed(SEED)
os.environ.setdefault('PYTHONHASHSEED', str(SEED))
os.environ.setdefault('OMP_NUM_THREADS', '4')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '4')
os.environ.setdefault('MKL_NUM_THREADS', '4')
os.environ.setdefault('VECLIB_MAXIMUM_THREADS', '4')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '4')

from sklearn.base import clone
from sklearn.model_selection import KFold, StratifiedKFold, RepeatedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, StandardScaler, QuantileTransformer, PowerTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import pairwise_distances
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge, BayesianRidge, HuberRegressor
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.utils.class_weight import compute_sample_weight

try:
    from scipy.special import expit, logit
    from scipy.optimize import minimize
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False
    def expit(x):
        x = np.asarray(x, dtype=float)
        return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))
    def logit(p):
        p = np.clip(np.asarray(p, dtype=float), 1e-6, 1-1e-6)
        return np.log(p / (1 - p))

try:
    from xgboost import XGBClassifier, XGBRegressor
    XGB_OK = True
except Exception:
    XGB_OK = False

try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    LGB_OK = True
except Exception:
    LGB_OK = False

try:
    from catboost import CatBoostClassifier, CatBoostRegressor
    CAT_OK = True
except Exception:
    CAT_OK = False

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_OK = True
    torch.set_num_threads(2)
except Exception:
    TORCH_OK = False

ID_COL = 'event_id'
TIME_COL = 'time_to_hit_hours'
EVENT_COL = 'event'
HORIZONS = [12, 24, 48]
FULL_HORIZONS = [12, 24, 48, 72]
PROB_COLS = [f'prob_{h}h' for h in FULL_HORIZONS]
REQUIRED_COLS = [ID_COL] + PROB_COLS
FORCE_PROB_72_ONE = True
EPS = 1e-7

# -------------------------- data location --------------------------
def locate_data():
    roots = []
    for r in ['/kaggle/input', '/kaggle/working', '.', '/mnt/data']:
        if os.path.isdir(r):
            roots.append(r)
    dirs = []
    for root in roots:
        for cur, _, files in os.walk(root):
            fl = set(files)
            if {'train.csv', 'test.csv', 'sample_submission.csv'}.issubset(fl):
                score = 0
                low = cur.lower()
                if 'wids' in low or 'global' in low or 'datathon' in low or 'dathon' in low:
                    score += 10
                if 'samplecsv' in low or 'sub' in low:
                    score -= 5
                score -= len(cur) / 1000
                dirs.append((score, cur))
    if not dirs:
        raise FileNotFoundError('Could not locate train.csv, test.csv, and sample_submission.csv under /kaggle/input.')
    dirs = sorted(dirs, reverse=True)
    d = dirs[0][1]
    return os.path.join(d, 'train.csv'), os.path.join(d, 'test.csv'), os.path.join(d, 'sample_submission.csv')

train_path, test_path, sample_path = locate_data()
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_path)

assert ID_COL in train.columns and ID_COL in test.columns and ID_COL in sample_sub.columns
assert TIME_COL in train.columns and EVENT_COL in train.columns

sample_ids = sample_sub[ID_COL].astype(str).tolist()
# Preserve the sample order exactly.
test = test.set_index(ID_COL).loc[sample_sub[ID_COL].values].reset_index()

feature_cols_raw = [c for c in test.columns if c != ID_COL]
# Drop accidental target columns if present.
feature_cols_raw = [c for c in feature_cols_raw if c not in [TIME_COL, EVENT_COL]]

time = train[TIME_COL].astype(float).to_numpy()
event = train[EVENT_COL].astype(int).to_numpy()
n_train = len(train)
n_test = len(test)

print('Input directory:', os.path.dirname(train_path))
print('Train/test:', train.shape, test.shape)
print('Events/censored:', int(event.sum()), int((1-event).sum()))

# -------------------------- feature engineering --------------------------
def _num(s, index=None, default=0.0):
    if isinstance(s, pd.Series):
        return pd.to_numeric(s, errors='coerce').astype(float)
    if index is None:
        return pd.Series(default)
    return pd.Series(np.full(len(index), default, dtype=float), index=index)

def add_engineered_features(df):
    out = df.copy()
    idx = out.index
    for c in list(out.columns):
        if c == ID_COL:
            continue
        out[c] = pd.to_numeric(out[c], errors='coerce')

    def g(c, default=0.0):
        if c in out.columns:
            return pd.to_numeric(out[c], errors='coerce').astype(float)
        return pd.Series(np.full(len(out), default, dtype=float), index=idx)

    # Missingness flags for rare but informative data gaps.
    for c in list(out.columns):
        if c != ID_COL and out[c].isna().any():
            out[c + '_isna'] = out[c].isna().astype(float)

    pi = math.pi
    area_first = g('area_first_ha')
    area_growth = g('area_growth_abs_0_5h')
    rel_growth = g('area_growth_rel_0_5h')
    area_final = area_first + area_growth
    area_final_alt = area_first * (1.0 + rel_growth.clip(lower=-0.95))
    area_final = area_final.where(np.isfinite(area_final) & (area_final > 0), area_final_alt)
    area_final = area_final.clip(lower=0)
    radius_first = np.sqrt(np.maximum(area_first, 0) * 10000.0 / pi)
    radius_final = np.sqrt(np.maximum(area_final, 0) * 10000.0 / pi)

    dist = g('dist_min_ci_0_5h')
    dist_change = g('dist_change_ci_0_5h')
    dist_slope = g('dist_slope_ci_0_5h')
    closing = g('closing_speed_m_per_h')
    closing_abs = g('closing_speed_abs_m_per_h')
    projected = g('projected_advance_m')
    dist_accel = g('dist_accel_m_per_h2')
    fit_r2 = g('dist_fit_r2_0_5h')
    radial = g('radial_growth_m')
    radial_rate = g('radial_growth_rate_m_per_h')
    centroid_disp = g('centroid_displacement_m')
    centroid_speed = g('centroid_speed_m_per_h')
    alignment = g('alignment_cos')
    alignment_abs = g('alignment_abs')
    along = g('along_track_speed')
    cross = g('cross_track_component')
    dt = g('dt_first_last_0_5h')
    nperi = g('num_perimeters_0_5h')
    lowres = g('low_temporal_resolution_0_5h')

    out['area_final_ha_est'] = area_final
    out['radius_first_m_est'] = radius_first
    out['radius_final_m_est'] = radius_final
    out['radius_growth_est'] = radius_final - radius_first
    out['dist_km'] = dist / 1000.0
    out['dist_over_5km_m'] = dist - 5000.0
    out['dist_over_10km_m'] = dist - 10000.0
    out['dist_over_20km_m'] = dist - 20000.0
    out['inside_5km_early'] = (dist <= 5000.0).astype(float)
    out['near_7p5km'] = (dist <= 7500.0).astype(float)
    out['near_10km'] = (dist <= 10000.0).astype(float)
    out['near_15km'] = (dist <= 15000.0).astype(float)
    out['near_25km'] = (dist <= 25000.0).astype(float)
    out['inv_dist_1km'] = 1.0 / (dist + 1000.0)
    out['inv_dist_5km'] = 1.0 / (dist + 5000.0)
    out['sqrt_dist'] = np.sqrt(np.maximum(dist, 0))
    out['log1p_dist'] = np.log1p(np.maximum(dist, 0))

    close_pos = np.maximum(closing, 0)
    close_reliable = np.maximum(closing, 0) * np.clip(fit_r2.fillna(0), 0, 1)
    radial_pos = np.maximum(radial_rate, 0)
    along_pos = np.maximum(along, 0)
    effective_speed_1 = np.maximum(close_pos + radial_pos, 0)
    effective_speed_2 = np.maximum(0.65 * close_pos + 0.35 * along_pos + 0.75 * radial_pos, 0)
    effective_speed_3 = np.maximum(close_reliable + 0.50 * radial_pos, 0)

    out['closing_positive_mph'] = close_pos
    out['closing_reliable_mph'] = close_reliable
    out['radial_positive_mph'] = radial_pos
    out['along_positive_mph'] = along_pos
    out['effective_speed_close_radial'] = effective_speed_1
    out['effective_speed_blend'] = effective_speed_2
    out['effective_speed_r2'] = effective_speed_3
    out['closing_per_km_distance'] = close_pos / (dist / 1000.0 + 1.0)
    out['radial_per_km_distance'] = radial_pos / (dist / 1000.0 + 1.0)
    out['centroid_per_km_distance'] = np.maximum(centroid_speed, 0) / (dist / 1000.0 + 1.0)
    out['projected_per_distance'] = projected / (dist + 1000.0)
    out['closing_vs_abs'] = closing / (closing_abs + 1.0)
    out['along_vs_centroid'] = along / (centroid_speed.abs() + 1.0)
    out['cross_vs_along_abs'] = cross.abs() / (along.abs() + 1.0)
    out['alignment_x_closing'] = alignment * close_pos
    out['alignmentabs_x_closing'] = alignment_abs * close_pos
    out['alignment_x_radial'] = alignment * radial_pos
    out['distance_x_alignment'] = dist * alignment
    out['distance_x_lowres'] = dist * lowres
    out['growth_x_near'] = np.log1p(np.maximum(area_growth, 0)) * (dist <= 15000).astype(float)
    out['growth_rate_x_alignment'] = g('area_growth_rate_ha_per_h') * alignment
    out['radial_rate_x_alignment'] = radial_rate * alignment
    out['accel_x_closing'] = dist_accel * np.sign(closing.replace(0, np.nan)).fillna(0)
    out['fitr2_x_closing'] = np.clip(fit_r2, 0, 1) * closing
    out['lowres_x_closing'] = lowres * closing
    out['nperi_per_hour'] = nperi / (dt.clip(lower=0.05) + 0.05)
    out['dt_x_growth_rate'] = dt * g('area_growth_rate_ha_per_h')
    out['dt_x_closing'] = dt * closing

    # Linear arrival proxies: time from the end of the 5h feature window to a 5km buffer.
    def t_arrive(speed, extra_radius=0.0):
        numer = np.maximum(dist - 5000.0 - extra_radius, 0.0)
        return np.where(speed > 1.0, numer / np.maximum(speed, 1.0), 999.0)

    out['t_centroid_5km_close'] = t_arrive(close_pos, 0.0)
    out['t_perimeter_5km_close_radial'] = t_arrive(effective_speed_1, radius_final)
    out['t_perimeter_5km_blend'] = t_arrive(effective_speed_2, radius_final)
    out['t_perimeter_5km_r2'] = t_arrive(effective_speed_3, radius_final)
    out['t_perimeter_5km_along'] = t_arrive(np.maximum(along_pos + radial_pos, 0), radius_final)

    for H in FULL_HORIZONS:
        h = float(H)
        center_margin = 5000.0 - (dist - close_pos * h)
        perim_margin1 = 5000.0 + radius_final + radial_pos * h - (dist - close_pos * h)
        perim_margin2 = 5000.0 + radius_final + radial_pos * h - (dist - along_pos * h)
        out[f'margin_center_{H}h'] = center_margin
        out[f'margin_perim_{H}h'] = perim_margin1
        out[f'margin_along_perim_{H}h'] = perim_margin2
        out[f'anchor_center_{H}h_slow'] = expit(center_margin / 2200.0)
        out[f'anchor_center_{H}h_fast'] = expit(center_margin / 1100.0)
        out[f'anchor_perim_{H}h_slow'] = expit(perim_margin1 / 2600.0)
        out[f'anchor_perim_{H}h_fast'] = expit(perim_margin1 / 1300.0)
        out[f'anchor_along_perim_{H}h'] = expit(perim_margin2 / 2200.0)
        out[f'time_gap_center_{H}h'] = h - out['t_centroid_5km_close']
        out[f'time_gap_perim_{H}h'] = h - out['t_perimeter_5km_close_radial']

    # Temporal cyclic encodings.
    if 'event_start_hour' in out.columns:
        hour = g('event_start_hour')
        out['hour_sin'] = np.sin(2 * pi * hour / 24.0)
        out['hour_cos'] = np.cos(2 * pi * hour / 24.0)
        out['is_afternoon'] = ((hour >= 12) & (hour <= 18)).astype(float)
        out['is_night'] = ((hour <= 5) | (hour >= 21)).astype(float)
    if 'event_start_dayofweek' in out.columns:
        dow = g('event_start_dayofweek')
        out['dow_sin'] = np.sin(2 * pi * dow / 7.0)
        out['dow_cos'] = np.cos(2 * pi * dow / 7.0)
        out['is_weekend'] = (dow >= 5).astype(float)
    if 'event_start_month' in out.columns:
        month = g('event_start_month')
        out['month_sin'] = np.sin(2 * pi * (month - 1) / 12.0)
        out['month_cos'] = np.cos(2 * pi * (month - 1) / 12.0)
        out['peak_fire_season'] = month.isin([6, 7, 8, 9, 10]).astype(float)
        out['late_season'] = month.isin([9, 10, 11]).astype(float)

    # Generic bounded transforms for skewed physical columns.
    skew_cols = [
        'area_first_ha', 'area_growth_abs_0_5h', 'area_growth_rate_ha_per_h',
        'radial_growth_m', 'radial_growth_rate_m_per_h', 'centroid_displacement_m',
        'centroid_speed_m_per_h', 'dist_min_ci_0_5h', 'dist_std_ci_0_5h',
        'closing_speed_abs_m_per_h', 'projected_advance_m', 'cross_track_component',
        'along_track_speed', 'dist_accel_m_per_h2'
    ]
    for c in skew_cols:
        if c in out.columns:
            x = pd.to_numeric(out[c], errors='coerce').astype(float)
            out[f'{c}_signed_log1p'] = np.sign(x) * np.log1p(np.abs(x))
            out[f'{c}_sqrt_abs'] = np.sqrt(np.abs(x))

    # Robust ratios with distance/growth.
    out['radius_final_per_dist'] = radius_final / (dist + 1000.0)
    out['radius_growth_per_dist'] = (radius_final - radius_first) / (dist + 1000.0)
    out['area_growth_per_initial_plus1'] = area_growth / (area_first.abs() + 1.0)
    out['area_final_per_dist2'] = area_final / ((dist / 1000.0 + 1.0) ** 2)
    out['spread_energy_proxy'] = np.log1p(np.maximum(area_final, 0)) * np.log1p(np.maximum(close_pos + radial_pos + centroid_speed, 0))
    out['urgency_physics_score'] = (
        1.5 * out['anchor_perim_24h_slow'] +
        2.2 * out['anchor_perim_48h_slow'] +
        0.8 * out['anchor_center_24h_slow'] +
        0.6 * out['near_10km'] +
        0.25 * np.clip(alignment, -1, 1)
    )

    out = out.replace([np.inf, -np.inf], np.nan)
    return out

all_base = pd.concat([
    train[[ID_COL] + feature_cols_raw].copy(),
    test[[ID_COL] + feature_cols_raw].copy()
], axis=0, ignore_index=True)
all_feat = add_engineered_features(all_base)

# Keep only numeric columns except the id.
feature_cols_all = [c for c in all_feat.columns if c != ID_COL]
all_feat[feature_cols_all] = all_feat[feature_cols_all].apply(pd.to_numeric, errors='coerce')
# Remove columns that are entirely missing or constant in train+test.
keep_cols = []
for c in feature_cols_all:
    x = all_feat[c].to_numpy(dtype=float)
    if np.all(~np.isfinite(x)):
        continue
    finite = x[np.isfinite(x)]
    if finite.size == 0:
        continue
    if np.nanstd(finite) < 1e-12:
        continue
    keep_cols.append(c)
feature_cols_all = keep_cols

X_all_df = all_feat.iloc[:n_train][feature_cols_all].reset_index(drop=True)
T_all_df = all_feat.iloc[n_train:][feature_cols_all].reset_index(drop=True)

# Original and physics-focused feature subsets for diversity.
orig_cols = [c for c in feature_cols_raw if c in feature_cols_all]
phys_tokens = [
    'dist', 'closing', 'projected', 'advance', 'radial', 'centroid', 'align', 'track',
    'area', 'growth', 'radius', 'margin', 'anchor', 'time_gap', 't_perimeter', 't_centroid',
    'speed', 'slope', 'accel', 'fitr2', 'near', 'inside', 'urgency', 'lowres', 'nperi', 'dt_'
]
phys_cols = [c for c in feature_cols_all if any(tok in c.lower() for tok in phys_tokens)]
if len(phys_cols) < 12:
    phys_cols = feature_cols_all
lowdim_cols = []
for c in [
    'dist_min_ci_0_5h', 'dist_km', 'dist_over_5km_m', 'closing_speed_m_per_h',
    'closing_positive_mph', 'projected_advance_m', 'radial_growth_rate_m_per_h',
    'radial_positive_mph', 'alignment_cos', 'alignment_abs', 'along_track_speed',
    'cross_track_component', 'area_first_ha', 'area_growth_abs_0_5h', 'area_growth_rate_ha_per_h',
    'radius_final_m_est', 't_perimeter_5km_close_radial', 't_centroid_5km_close',
    'anchor_perim_24h_slow', 'anchor_perim_48h_slow', 'anchor_center_24h_slow',
    'event_start_month', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos'
]:
    if c in feature_cols_all:
        lowdim_cols.append(c)
if len(lowdim_cols) < 8:
    lowdim_cols = phys_cols[:min(40, len(phys_cols))]

FEATURE_SETS = {
    'all': feature_cols_all,
    'orig': orig_cols if len(orig_cols) >= 5 else feature_cols_all,
    'phys': phys_cols,
    'low': lowdim_cols,
}
print('Feature counts:', {k: len(v) for k, v in FEATURE_SETS.items()})

# -------------------------- metric helpers --------------------------
def known_mask_for_horizon(t, e, H):
    t = np.asarray(t, dtype=float)
    e = np.asarray(e, dtype=int)
    return (e == 1) | (t >= H - 1e-9)

def binary_label_for_horizon(t, e, H):
    t = np.asarray(t, dtype=float)
    e = np.asarray(e, dtype=int)
    return ((e == 1) & (t <= H + 1e-9)).astype(int)

def enforce_monotonic(P, force_72=FORCE_PROB_72_ONE):
    P = np.asarray(P, dtype=float).copy()
    if P.ndim == 1:
        P = P.reshape(-1, 4)
    P = np.nan_to_num(P, nan=0.0, posinf=1.0, neginf=0.0)
    P = np.clip(P, 0.0, 1.0)
    P = np.maximum.accumulate(P, axis=1)
    if force_72:
        P[:, 3] = 1.0
    P[:, :3] = np.clip(P[:, :3], 1e-5, 1.0 - 1e-5)
    P = np.maximum.accumulate(P, axis=1)
    return np.clip(P, 0.0, 1.0)

def weighted_brier(P, t=time, e=event):
    P = enforce_monotonic(P)
    weights = {24: 0.3, 48: 0.4, 72: 0.3}
    col_idx = {12: 0, 24: 1, 48: 2, 72: 3}
    out = 0.0
    wsum = 0.0
    for H, w in weights.items():
        mask = known_mask_for_horizon(t, e, H)
        if mask.sum() == 0:
            continue
        y = binary_label_for_horizon(t, e, H)
        b = float(np.mean((P[mask, col_idx[H]] - y[mask]) ** 2))
        out += w * b
        wsum += w
    return out / max(wsum, 1e-12)

def concordance_index(P, t=time, e=event, risk_weights=(0.18, 0.34, 0.48, 0.00)):
    P = enforce_monotonic(P)
    risk = P @ np.asarray(risk_weights, dtype=float)
    n = len(t)
    concordant = 0.0
    comparable = 0.0
    for i in range(n):
        if e[i] != 1:
            continue
        ti = t[i]
        ri = risk[i]
        for j in range(n):
            if i == j:
                continue
            # Event i is known to occur before j's event/censoring time.
            if ti < t[j] - 1e-9:
                comparable += 1.0
                if ri > risk[j] + 1e-12:
                    concordant += 1.0
                elif abs(ri - risk[j]) <= 1e-12:
                    concordant += 0.5
    if comparable == 0:
        return 0.5
    return concordant / comparable

def hybrid_score(P, t=time, e=event):
    P = enforce_monotonic(P)
    best_c = 0.5
    # The official C-index risk aggregator is not exposed. Use a small family and keep the most plausible OOF surrogate.
    for rw in [(0.22,0.34,0.44,0.0), (0.15,0.35,0.50,0.0), (0.33,0.33,0.34,0.0), (0.10,0.30,0.60,0.0)]:
        best_c = max(best_c, concordance_index(P, t, e, rw))
    wb = weighted_brier(P, t, e)
    return 0.3 * best_c + 0.7 * (1.0 - wb)

def score_report(P, name=''):
    P = enforce_monotonic(P)
    rep = {
        'name': name,
        'hybrid': hybrid_score(P),
        'cindex': max(concordance_index(P, risk_weights=rw) for rw in [(0.22,0.34,0.44,0.0),(0.15,0.35,0.50,0.0),(0.33,0.33,0.34,0.0)]),
        'wbrier': weighted_brier(P)
    }
    for H, j in zip(FULL_HORIZONS, range(4)):
        mask = known_mask_for_horizon(time, event, H)
        if mask.sum() > 0:
            y = binary_label_for_horizon(time, event, H)
            rep[f'brier_{H}'] = float(np.mean((P[mask, j] - y[mask]) ** 2))
            rep[f'meanp_{H}'] = float(np.mean(P[:, j]))
    return rep

# -------------------------- utility functions --------------------------
def safe_predict_proba(model, X):
    if hasattr(model, 'predict_proba'):
        p = model.predict_proba(X)
        if p.ndim == 2 and p.shape[1] >= 2:
            return p[:, 1].astype(float)
        return np.asarray(p).reshape(-1).astype(float)
    if hasattr(model, 'decision_function'):
        z = model.decision_function(X)
        return expit(np.asarray(z, dtype=float).reshape(-1))
    z = model.predict(X)
    return np.asarray(z, dtype=float).reshape(-1)

def _fit_with_optional_weight(estimator, X, y, sample_weight=None):
    if sample_weight is None:
        estimator.fit(X, y)
        return estimator
    try:
        estimator.fit(X, y, sample_weight=sample_weight)
        return estimator
    except Exception:
        pass
    # sklearn Pipeline requires stepname__sample_weight for the final estimator.
    if hasattr(estimator, 'steps') and len(estimator.steps) > 0:
        step_name = estimator.steps[-1][0]
        try:
            estimator.fit(X, y, **{f'{step_name}__sample_weight': sample_weight})
            return estimator
        except Exception:
            pass
    estimator.fit(X, y)
    return estimator

def model_fit_predict(model, X_tr, y_tr, X_val, X_te, sample_weight=None):
    m = clone(model)
    try:
        m = _fit_with_optional_weight(m, X_tr, y_tr, sample_weight)
    except Exception:
        m = clone(model)
        m.fit(X_tr, y_tr)
    pv = safe_predict_proba(m, X_val)
    pt = safe_predict_proba(m, X_te)
    return pv, pt

def reg_fit_predict(model, X_tr, y_tr, X_val, X_te, sample_weight=None):
    m = clone(model)
    try:
        m = _fit_with_optional_weight(m, X_tr, y_tr, sample_weight)
    except Exception:
        m = clone(model)
        m.fit(X_tr, y_tr)
    pv = np.asarray(m.predict(X_val), dtype=float).reshape(-1)
    pt = np.asarray(m.predict(X_te), dtype=float).reshape(-1)
    return pv, pt

def make_folds(seed=SEED, n_splits=5, n_repeats=5):
    # Stratify by event and broad time bins to stabilize rare early hits.
    bins = np.digitize(time, [12, 24, 48, 72])
    strat = event.astype(str) + '_' + bins.astype(str)
    counts = pd.Series(strat).value_counts()
    if counts.min() >= n_splits:
        folds = []
        for r in range(n_repeats):
            skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed + 77*r)
            for tr_idx, va_idx in skf.split(np.zeros(n_train), strat):
                folds.append((tr_idx, va_idx))
        return folds
    rkf = RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=seed)
    return list(rkf.split(np.zeros(n_train)))

FOLDS = make_folds(SEED, 5, 6)
FAST_FOLDS = make_folds(SEED + 111, 5, 3)

# Test-density weights from adversarial validation. Fully transductive, label-free, and allowed.
def compute_test_density_weights():
    cols = FEATURE_SETS['all']
    X = pd.concat([X_all_df[cols], T_all_df[cols]], axis=0, ignore_index=True)
    y_dom = np.r_[np.zeros(n_train), np.ones(n_test)]
    prep = make_pipeline(SimpleImputer(strategy='median'), RobustScaler())
    Xp = prep.fit_transform(X)
    models = []
    models.append(ExtraTreesClassifier(n_estimators=350, min_samples_leaf=4, max_features='sqrt', random_state=SEED+1, n_jobs=-1))
    models.append(RandomForestClassifier(n_estimators=300, min_samples_leaf=5, max_features='sqrt', random_state=SEED+2, n_jobs=-1))
    dom_p = np.zeros(n_train)
    for m in models:
        try:
            m.fit(Xp, y_dom)
            p = m.predict_proba(Xp[:n_train])[:, 1]
            dom_p += p / len(models)
        except Exception:
            dom_p += np.full(n_train, n_test/(n_train+n_test)) / len(models)
    odds = dom_p / np.clip(1 - dom_p, 0.05, None)
    odds = np.clip(odds, np.quantile(odds, 0.05), np.quantile(odds, 0.95))
    odds = odds / np.mean(odds)
    return np.clip(odds, 0.45, 2.25)

TEST_DENSITY_W = compute_test_density_weights()
print('Test-density weights:', float(TEST_DENSITY_W.min()), float(TEST_DENSITY_W.max()), float(TEST_DENSITY_W.mean()))

# -------------------------- candidates --------------------------
oof_candidates = []
test_candidates = []
candidate_names = []

# cache numeric matrices by feature set and preprocessing name.
def get_matrix(cols):
    return X_all_df[cols].copy(), T_all_df[cols].copy()

def add_candidate(name, oof, test_pred, do_calibrated=True):
    oof = enforce_monotonic(oof)
    test_pred = enforce_monotonic(test_pred)
    if np.any(~np.isfinite(test_pred)) or np.any(~np.isfinite(oof)):
        return
    oof_candidates.append(oof)
    test_candidates.append(test_pred)
    candidate_names.append(name)
    if do_calibrated:
        coof, ctest = calibrate_candidate(oof, test_pred)
        if np.nanmean(np.abs(coof - oof)) > 1e-7:
            oof_candidates.append(enforce_monotonic(coof))
            test_candidates.append(enforce_monotonic(ctest))
            candidate_names.append(name + '_cal')

def prior_candidate():
    P = np.zeros((n_train, 4), dtype=float)
    Q = np.zeros((n_test, 4), dtype=float)
    for j, H in enumerate(FULL_HORIZONS):
        if H == 72 and FORCE_PROB_72_ONE:
            p = 1.0
        else:
            mask = known_mask_for_horizon(time, event, H)
            y = binary_label_for_horizon(time, event, H)
            # Beta smoothing, slightly conservative for calibration.
            p = (y[mask].sum() + 0.7) / (mask.sum() + 1.4)
        P[:, j] = p
        Q[:, j] = p
    return enforce_monotonic(P), enforce_monotonic(Q)

# Calibration using horizon-specific cross-fitted isotonic/logistic blends.
def _fit_logistic_calibrator(x, y):
    x = np.clip(np.asarray(x, dtype=float), 1e-5, 1-1e-5)
    y = np.asarray(y, dtype=int)
    if len(np.unique(y)) < 2:
        p = float(np.mean(y)) if len(y) else 0.0
        return lambda z: np.full(len(np.asarray(z).reshape(-1)), p, dtype=float)
    lr = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
    lr.fit(logit(x).reshape(-1, 1), y)
    return lambda z: lr.predict_proba(logit(np.clip(np.asarray(z, dtype=float), 1e-5, 1-1e-5)).reshape(-1, 1))[:, 1]

def _calibrate_1d(oof_raw, test_raw, H, blend_grid=(0.0, 0.15, 0.30, 0.45, 0.60, 0.80)):
    oof_raw = np.clip(np.asarray(oof_raw, dtype=float), 1e-5, 1-1e-5)
    test_raw = np.clip(np.asarray(test_raw, dtype=float), 1e-5, 1-1e-5)
    mask = known_mask_for_horizon(time, event, H)
    y = binary_label_for_horizon(time, event, H)
    if mask.sum() < 12 or len(np.unique(y[mask])) < 2:
        return oof_raw, test_raw
    # Fit calibrators on all OOF predictions, then choose a conservative blend by Brier.
    preds = []
    preds_t = []
    try:
        iso = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
        iso.fit(oof_raw[mask], y[mask])
        preds.append(np.clip(iso.predict(oof_raw), 1e-5, 1-1e-5))
        preds_t.append(np.clip(iso.predict(test_raw), 1e-5, 1-1e-5))
    except Exception:
        pass
    try:
        f = _fit_logistic_calibrator(oof_raw[mask], y[mask])
        preds.append(np.clip(f(oof_raw), 1e-5, 1-1e-5))
        preds_t.append(np.clip(f(test_raw), 1e-5, 1-1e-5))
    except Exception:
        pass
    if not preds:
        return oof_raw, test_raw
    cal_oof = np.mean(preds, axis=0)
    cal_test = np.mean(preds_t, axis=0)
    raw_b = float(np.mean((oof_raw[mask] - y[mask]) ** 2))
    best = (raw_b, 0.0)
    for lam in blend_grid:
        z = (1-lam) * oof_raw + lam * cal_oof
        b = float(np.mean((z[mask] - y[mask]) ** 2))
        # penalty prevents tiny train-set gains from over-calibrating.
        penalized = b + 0.00035 * lam * lam
        if penalized < best[0] + 1e-12:
            best = (penalized, lam)
    lam = best[1]
    return (1-lam) * oof_raw + lam * cal_oof, (1-lam) * test_raw + lam * cal_test

def calibrate_candidate(oof, test_pred):
    P = np.asarray(oof, dtype=float).copy()
    Q = np.asarray(test_pred, dtype=float).copy()
    for j, H in enumerate(HORIZONS):
        P[:, j], Q[:, j] = _calibrate_1d(P[:, j], Q[:, j], H)
    if FORCE_PROB_72_ONE:
        P[:, 3] = 1.0
        Q[:, 3] = 1.0
    return enforce_monotonic(P), enforce_monotonic(Q)

p0, q0 = prior_candidate()
add_candidate('prior_km_metric', p0, q0, do_calibrated=False)

# -------------------------- physics anchors --------------------------
def make_physics_candidates():
    tr = X_all_df
    te = T_all_df
    candidates = []

    def col(df, c, default=0.0):
        if c in df.columns:
            return pd.to_numeric(df[c], errors='coerce').fillna(default).to_numpy(dtype=float)
        return np.full(len(df), default, dtype=float)

    for name, params in {
        'phys_balanced': (0.55, 0.45, 2400.0, 1.00),
        'phys_fast': (0.75, 0.25, 1550.0, 1.15),
        'phys_slow': (0.40, 0.60, 3400.0, 0.85),
        'phys_center': (1.00, 0.00, 2200.0, 1.00),
        'phys_perimeter': (0.20, 0.80, 2500.0, 1.00),
        'phys_alignment': (0.55, 0.45, 2100.0, 1.00),
    }.items():
        a_center, a_perim, scale, speed_mult = params
        Ps = []
        for df in [tr, te]:
            dist = col(df, 'dist_min_ci_0_5h')
            closing = np.maximum(col(df, 'closing_speed_m_per_h'), 0)
            radial = np.maximum(col(df, 'radial_growth_rate_m_per_h'), 0)
            along = np.maximum(col(df, 'along_track_speed'), 0)
            align = np.clip(col(df, 'alignment_cos'), -1, 1)
            radius = np.maximum(col(df, 'radius_final_m_est'), 0)
            fit = np.clip(col(df, 'dist_fit_r2_0_5h', 0.5), 0, 1)
            growth = np.log1p(np.maximum(col(df, 'area_growth_abs_0_5h'), 0))
            near_prior = expit((18000.0 - dist) / 7000.0)
            P = np.zeros((len(df), 4), dtype=float)
            for j, H in enumerate(FULL_HORIZONS):
                if H == 72 and FORCE_PROB_72_ONE:
                    P[:, j] = 1.0
                    continue
                h = float(H)
                speed = speed_mult * (0.62 * closing + 0.23 * along + 0.15 * closing * fit + 0.60 * radial)
                if name == 'phys_alignment':
                    speed = speed * (0.82 + 0.35 * np.maximum(align, 0))
                margin_center = 5000.0 - (dist - speed * h)
                margin_perim = 5000.0 + radius + radial * h - (dist - speed * h)
                p_center = expit(margin_center / scale)
                p_perim = expit(margin_perim / (scale * 1.08))
                # Multiplicative prior helps avoid overconfident far-away anchors.
                p = a_center * p_center + a_perim * p_perim
                p = 0.88 * p + 0.12 * near_prior * expit((growth - 3.0) / 1.5)
                P[:, j] = p
            Ps.append(enforce_monotonic(P))
        candidates.append((name, Ps[0], Ps[1]))
    return candidates

for name, p, q in make_physics_candidates():
    add_candidate(name, p, q, do_calibrated=True)

# -------------------------- KNN censor-aware local Kaplan candidates --------------------------
def make_scaled_np(train_df, test_df, scaler_kind='robust'):
    imp = SimpleImputer(strategy='median')
    Xtr = imp.fit_transform(train_df)
    Xte = imp.transform(test_df)
    if scaler_kind == 'quantile':
        scaler = QuantileTransformer(n_quantiles=min(64, len(Xtr)), output_distribution='normal', random_state=SEED)
    elif scaler_kind == 'standard':
        scaler = StandardScaler()
    else:
        scaler = RobustScaler()
    Xtr = scaler.fit_transform(Xtr)
    Xte = scaler.transform(Xte)
    Xtr = np.nan_to_num(Xtr, nan=0.0, posinf=0.0, neginf=0.0)
    Xte = np.nan_to_num(Xte, nan=0.0, posinf=0.0, neginf=0.0)
    return Xtr, Xte

def knn_predict_for_horizon(Xref, tref, eref, Xq, H, k=21, power=1.0, global_prior=None):
    mask_known = known_mask_for_horizon(tref, eref, H)
    y = binary_label_for_horizon(tref, eref, H).astype(float)
    if global_prior is None:
        global_prior = (y[mask_known].sum() + 0.7) / (mask_known.sum() + 1.4) if mask_known.sum() else 0.5
    if Xref.shape[0] == 0:
        return np.full(Xq.shape[0], global_prior, dtype=float)
    d = pairwise_distances(Xq, Xref, metric='euclidean', n_jobs=-1)
    kk = min(k, Xref.shape[0])
    idx = np.argpartition(d, kth=kk-1, axis=1)[:, :kk]
    out = np.zeros(Xq.shape[0], dtype=float)
    for i in range(Xq.shape[0]):
        ii = idx[i]
        di = d[i, ii]
        known = mask_known[ii]
        if known.sum() < 2:
            out[i] = global_prior
            continue
        ii = ii[known]
        di = di[known]
        w = 1.0 / np.maximum(di, 1e-4) ** power
        # Conservative shrink to global prior when the local effective sample size is low.
        ess = (w.sum() ** 2) / np.sum(w ** 2)
        local = float(np.sum(w * y[ii]) / np.sum(w))
        shrink = min(1.0, ess / 12.0)
        out[i] = shrink * local + (1 - shrink) * global_prior
    return np.clip(out, 1e-5, 1-1e-5)

def make_knn_candidate(cols_name, cols, scaler_kind='robust', k=21, power=1.0, folds=FAST_FOLDS):
    Xdf, Tdf = get_matrix(cols)
    oof = np.zeros((n_train, 4), dtype=float)
    test_acc = np.zeros((n_test, 4), dtype=float)
    counts = np.zeros(n_train, dtype=int)
    # Full scaled matrix for test full-train estimates.
    Xfull, Tfull = make_scaled_np(Xdf, Tdf, scaler_kind)
    for j, H in enumerate(HORIZONS):
        y = binary_label_for_horizon(time, event, H)
        mask = known_mask_for_horizon(time, event, H)
        gp = (y[mask].sum() + 0.7) / (mask.sum() + 1.4) if mask.sum() else 0.5
        test_acc[:, j] = knn_predict_for_horizon(Xfull, time, event, Tfull, H, k=k, power=power, global_prior=gp)
    if FORCE_PROB_72_ONE:
        test_acc[:, 3] = 1.0
    else:
        H = 72
        y = binary_label_for_horizon(time, event, H); mask = known_mask_for_horizon(time, event, H)
        gp = (y[mask].sum() + 0.7) / (mask.sum() + 1.4) if mask.sum() else 0.5
        test_acc[:, 3] = knn_predict_for_horizon(Xfull, time, event, Tfull, H, k=k, power=power, global_prior=gp)

    for tr_idx, va_idx in folds:
        Xtr_df, Xva_df = Xdf.iloc[tr_idx], Xdf.iloc[va_idx]
        Xtr, Xva = make_scaled_np(Xtr_df, Xva_df, scaler_kind)
        for j, H in enumerate(HORIZONS):
            ytr = binary_label_for_horizon(time[tr_idx], event[tr_idx], H)
            mtr = known_mask_for_horizon(time[tr_idx], event[tr_idx], H)
            gp = (ytr[mtr].sum() + 0.7) / (mtr.sum() + 1.4) if mtr.sum() else 0.5
            oof[va_idx, j] += knn_predict_for_horizon(Xtr, time[tr_idx], event[tr_idx], Xva, H, k=k, power=power, global_prior=gp)
        oof[va_idx, 3] += 1.0 if FORCE_PROB_72_ONE else 0.0
        counts[va_idx] += 1
    counts = np.maximum(counts, 1)
    oof = oof / counts[:, None]
    if FORCE_PROB_72_ONE:
        oof[:, 3] = 1.0
    return enforce_monotonic(oof), enforce_monotonic(test_acc)

knn_specs = [
    ('low', 'robust', 7, 1.2), ('low', 'robust', 13, 1.0), ('low', 'standard', 19, 1.0),
    ('phys', 'robust', 11, 1.0), ('phys', 'robust', 23, 0.8), ('phys', 'quantile', 17, 1.0),
    ('all', 'robust', 15, 0.8), ('all', 'quantile', 25, 0.7), ('orig', 'robust', 11, 1.1),
]
for fs, sc, k, pw in knn_specs:
    try:
        p, q = make_knn_candidate(fs, FEATURE_SETS[fs], sc, k, pw, FAST_FOLDS)
        add_candidate(f'knn_{fs}_{sc}_k{k}_p{pw}', p, q, do_calibrated=True)
        print('KNN candidate:', candidate_names[-1], score_report(oof_candidates[-1])['hybrid'])
    except Exception as ex:
        print('KNN skipped:', fs, sc, k, ex)

# -------------------------- binary classifiers per horizon --------------------------
def build_classifier_specs():
    specs = []
    # Tree ensembles: high variance reduction, strong ranking on tiny tabular data.
    for leaf, mf, name_suffix in [(2, 'sqrt', 'leaf2'), (4, 0.55, 'leaf4'), (7, 0.75, 'leaf7')]:
        specs.append((f'et_{name_suffix}', 'all', make_pipeline(
            SimpleImputer(strategy='median'),
            ExtraTreesClassifier(n_estimators=900, min_samples_leaf=leaf, max_features=mf, bootstrap=True,
                                 class_weight='balanced', random_state=SEED+leaf, n_jobs=-1)
        ), FOLDS, True))
    specs.append(('et_phys_deep', 'phys', make_pipeline(
        SimpleImputer(strategy='median'),
        ExtraTreesClassifier(n_estimators=1100, min_samples_leaf=2, max_features='sqrt', bootstrap=False,
                             class_weight='balanced', random_state=SEED+17, n_jobs=-1)
    ), FOLDS, True))
    specs.append(('rf_low', 'low', make_pipeline(
        SimpleImputer(strategy='median'),
        RandomForestClassifier(n_estimators=800, min_samples_leaf=3, max_features='sqrt', bootstrap=True,
                               class_weight='balanced_subsample', random_state=SEED+23, n_jobs=-1)
    ), FOLDS, True))
    specs.append(('rf_all_smooth', 'all', make_pipeline(
        SimpleImputer(strategy='median'),
        RandomForestClassifier(n_estimators=700, min_samples_leaf=7, max_features=0.55, bootstrap=True,
                               class_weight='balanced_subsample', random_state=SEED+24, n_jobs=-1)
    ), FOLDS, True))

    # Gradient boosting variants.
    specs.append(('gb_low', 'low', make_pipeline(
        SimpleImputer(strategy='median'),
        GradientBoostingClassifier(n_estimators=250, learning_rate=0.025, max_depth=2, min_samples_leaf=5,
                                   subsample=0.85, random_state=SEED+31)
    ), FOLDS, True))
    specs.append(('gb_phys', 'phys', make_pipeline(
        SimpleImputer(strategy='median'),
        GradientBoostingClassifier(n_estimators=320, learning_rate=0.018, max_depth=2, min_samples_leaf=6,
                                   subsample=0.80, random_state=SEED+32)
    ), FOLDS, True))
    specs.append(('hgb_low', 'low', make_pipeline(
        SimpleImputer(strategy='median'),
        HistGradientBoostingClassifier(max_iter=350, learning_rate=0.025, max_leaf_nodes=8, l2_regularization=0.10,
                                       min_samples_leaf=8, random_state=SEED+33)
    ), FOLDS, True))
    specs.append(('hgb_all', 'all', make_pipeline(
        SimpleImputer(strategy='median'),
        HistGradientBoostingClassifier(max_iter=300, learning_rate=0.022, max_leaf_nodes=10, l2_regularization=0.30,
                                       min_samples_leaf=10, random_state=SEED+34)
    ), FOLDS, True))

    # Smooth probabilistic baselines; useful for Brier after blending.
    specs.append(('logit_low_l2', 'low', make_pipeline(
        SimpleImputer(strategy='median'), RobustScaler(),
        LogisticRegression(C=0.38, penalty='l2', solver='lbfgs', max_iter=5000, class_weight='balanced')
    ), FOLDS, True))
    specs.append(('logit_phys_l1', 'phys', make_pipeline(
        SimpleImputer(strategy='median'), RobustScaler(),
        LogisticRegression(C=0.09, penalty='l1', solver='liblinear', max_iter=5000, class_weight='balanced')
    ), FOLDS, True))
    specs.append(('svc_low', 'low', make_pipeline(
        SimpleImputer(strategy='median'), RobustScaler(),
        SVC(C=0.75, gamma='scale', kernel='rbf', probability=True, class_weight='balanced', random_state=SEED+41)
    ), FAST_FOLDS, False))
    specs.append(('gnb_low', 'low', make_pipeline(
        SimpleImputer(strategy='median'), QuantileTransformer(n_quantiles=min(48, n_train), output_distribution='normal', random_state=SEED+42),
        GaussianNB(var_smoothing=1e-8)
    ), FAST_FOLDS, False))
    specs.append(('knnclf_low', 'low', make_pipeline(
        SimpleImputer(strategy='median'), RobustScaler(),
        KNeighborsClassifier(n_neighbors=13, weights='distance', p=2)
    ), FAST_FOLDS, False))

    if XGB_OK:
        specs.append(('xgb_low', 'low', XGBClassifier(
            n_estimators=650, max_depth=2, learning_rate=0.018, subsample=0.86, colsample_bytree=0.88,
            min_child_weight=2.0, reg_lambda=8.0, reg_alpha=0.10, objective='binary:logistic',
            eval_metric='logloss', random_state=SEED+51, n_jobs=2, verbosity=0
        ), FOLDS, True))
        specs.append(('xgb_phys', 'phys', XGBClassifier(
            n_estimators=520, max_depth=2, learning_rate=0.022, subsample=0.80, colsample_bytree=0.72,
            min_child_weight=2.5, reg_lambda=12.0, reg_alpha=0.25, objective='binary:logistic',
            eval_metric='logloss', random_state=SEED+52, n_jobs=2, verbosity=0
        ), FOLDS, True))
    if LGB_OK:
        specs.append(('lgb_low', 'low', LGBMClassifier(
            n_estimators=750, learning_rate=0.012, num_leaves=7, max_depth=3, min_child_samples=9,
            subsample=0.82, colsample_bytree=0.82, reg_alpha=0.10, reg_lambda=6.0,
            class_weight='balanced', random_state=SEED+61, n_jobs=2, verbose=-1
        ), FOLDS, True))
        specs.append(('lgb_phys', 'phys', LGBMClassifier(
            n_estimators=620, learning_rate=0.015, num_leaves=9, max_depth=3, min_child_samples=11,
            subsample=0.78, colsample_bytree=0.74, reg_alpha=0.20, reg_lambda=9.0,
            class_weight='balanced', random_state=SEED+62, n_jobs=2, verbose=-1
        ), FOLDS, True))
    if CAT_OK:
        specs.append(('cat_low', 'low', CatBoostClassifier(
            iterations=620, depth=2, learning_rate=0.018, l2_leaf_reg=12.0, loss_function='Logloss',
            eval_metric='Logloss', random_seed=SEED+71, verbose=False, allow_writing_files=False,
            bootstrap_type='Bayesian', bagging_temperature=0.8, random_strength=1.0
        ), FOLDS, True))
        specs.append(('cat_phys', 'phys', CatBoostClassifier(
            iterations=540, depth=3, learning_rate=0.016, l2_leaf_reg=18.0, loss_function='Logloss',
            eval_metric='Logloss', random_seed=SEED+72, verbose=False, allow_writing_files=False,
            bootstrap_type='Bayesian', bagging_temperature=1.2, random_strength=1.4
        ), FOLDS, True))
    return specs

def cv_binary_candidate(spec_name, fs_name, model, folds, use_density=True):
    cols = FEATURE_SETS[fs_name]
    Xdf, Tdf = get_matrix(cols)
    oof = np.zeros((n_train, 4), dtype=float)
    pred_test = np.zeros((n_test, 4), dtype=float)
    counts = np.zeros((n_train, 4), dtype=float)
    for j, H in enumerate(HORIZONS):
        y_all = binary_label_for_horizon(time, event, H)
        known_all = known_mask_for_horizon(time, event, H)
        test_fold_preds = []
        for fold_i, (tr_idx0, va_idx) in enumerate(folds):
            fit_idx = np.array([i for i in tr_idx0 if known_all[i]], dtype=int)
            if len(fit_idx) < 8 or len(np.unique(y_all[fit_idx])) < 2:
                prior = (y_all[known_all].sum()+0.7)/(known_all.sum()+1.4)
                oof[va_idx, j] += prior
                test_fold_preds.append(np.full(n_test, prior, dtype=float))
                counts[va_idx, j] += 1
                continue
            Xtr = Xdf.iloc[fit_idx]
            Xva = Xdf.iloc[va_idx]
            Xte = Tdf
            ytr = y_all[fit_idx]
            sw = compute_sample_weight(class_weight='balanced', y=ytr).astype(float)
            # Emphasize records whose covariates look like the test set and records with reliable follow-up.
            if use_density:
                sw *= TEST_DENSITY_W[fit_idx]
            # Earlier events matter for ranking; retain moderate effect to avoid hurting Brier.
            event_time_bonus = np.where(event[fit_idx] == 1, 1.0 + 0.20 * (72.0 - np.minimum(time[fit_idx], 72.0)) / 72.0, 1.0)
            sw *= event_time_bonus
            sw = sw / np.mean(sw)
            try:
                pv, pt = model_fit_predict(model, Xtr, ytr, Xva, Xte, sample_weight=sw)
            except Exception as ex:
                prior = (y_all[fit_idx].sum()+0.7)/(len(fit_idx)+1.4)
                pv = np.full(len(va_idx), prior, dtype=float)
                pt = np.full(n_test, prior, dtype=float)
            oof[va_idx, j] += pv
            test_fold_preds.append(pt)
            counts[va_idx, j] += 1
        pred_test[:, j] = np.mean(test_fold_preds, axis=0)
    counts = np.maximum(counts, 1)
    oof[:, :3] = oof[:, :3] / counts[:, :3]
    oof[:, 3] = 1.0 if FORCE_PROB_72_ONE else np.maximum(oof[:, 2], 0.5)
    pred_test[:, 3] = 1.0 if FORCE_PROB_72_ONE else np.maximum(pred_test[:, 2], 0.5)
    return enforce_monotonic(oof), enforce_monotonic(pred_test)

clf_specs = build_classifier_specs()
print('Classifier specs:', len(clf_specs), 'xgb/lgb/cat:', XGB_OK, LGB_OK, CAT_OK)
for spec_name, fs_name, model, folds, use_density in clf_specs:
    try:
        p, q = cv_binary_candidate(spec_name, fs_name, model, folds, use_density=use_density)
        add_candidate(f'bin_{spec_name}_{fs_name}', p, q, do_calibrated=True)
        rep = score_report(oof_candidates[-1], candidate_names[-1])
        print('BIN', candidate_names[-1], round(rep['hybrid'], 6), round(rep['wbrier'], 6), round(rep['cindex'], 6))
    except Exception as ex:
        print('BIN skipped:', spec_name, ex)
        gc.collect()

# -------------------------- urgency/ranking regressors converted to horizon probabilities --------------------------
def calibrate_risk_to_probs(risk_oof, risk_test, name_prefix):
    risk_oof = np.asarray(risk_oof, dtype=float).reshape(-1)
    risk_test = np.asarray(risk_test, dtype=float).reshape(-1)
    P = np.zeros((n_train, 4), dtype=float)
    Q = np.zeros((n_test, 4), dtype=float)
    # Normalize ranks for stability.
    ro = pd.Series(risk_oof).rank(method='average', pct=True).to_numpy(dtype=float)
    rt = pd.Series(np.r_[risk_oof, risk_test]).rank(method='average', pct=True).to_numpy(dtype=float)[n_train:]
    for j, H in enumerate(HORIZONS):
        mask = known_mask_for_horizon(time, event, H)
        y = binary_label_for_horizon(time, event, H)
        if mask.sum() < 12 or len(np.unique(y[mask])) < 2:
            prior = (y[mask].sum()+0.7)/(mask.sum()+1.4)
            P[:, j] = prior; Q[:, j] = prior; continue
        # Isotonic on ranks is intentionally monotone and robust for C-index.
        try:
            iso = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0, increasing=True)
            iso.fit(ro[mask], y[mask])
            p_iso = iso.predict(ro)
            q_iso = iso.predict(rt)
        except Exception:
            p_iso = np.full(n_train, np.mean(y[mask])); q_iso = np.full(n_test, np.mean(y[mask]))
        try:
            f = _fit_logistic_calibrator(ro[mask], y[mask])
            p_log = f(ro); q_log = f(rt)
        except Exception:
            p_log = p_iso; q_log = q_iso
        P[:, j] = 0.55 * p_iso + 0.45 * p_log
        Q[:, j] = 0.55 * q_iso + 0.45 * q_log
    P[:, 3] = 1.0 if FORCE_PROB_72_ONE else np.maximum(P[:, 2], 0.5)
    Q[:, 3] = 1.0 if FORCE_PROB_72_ONE else np.maximum(Q[:, 2], 0.5)
    return enforce_monotonic(P), enforce_monotonic(Q)

def build_regressor_specs():
    specs = []
    specs.append(('etr_risk_all', 'all', make_pipeline(
        SimpleImputer(strategy='median'),
        ExtraTreesRegressor(n_estimators=900, min_samples_leaf=3, max_features='sqrt', bootstrap=True, random_state=SEED+101, n_jobs=-1)
    ), FOLDS))
    specs.append(('etr_risk_low', 'low', make_pipeline(
        SimpleImputer(strategy='median'),
        ExtraTreesRegressor(n_estimators=800, min_samples_leaf=2, max_features=0.85, bootstrap=True, random_state=SEED+102, n_jobs=-1)
    ), FOLDS))
    specs.append(('rf_risk_phys', 'phys', make_pipeline(
        SimpleImputer(strategy='median'),
        RandomForestRegressor(n_estimators=650, min_samples_leaf=4, max_features='sqrt', bootstrap=True, random_state=SEED+103, n_jobs=-1)
    ), FOLDS))
    specs.append(('gbr_risk_low', 'low', make_pipeline(
        SimpleImputer(strategy='median'),
        GradientBoostingRegressor(n_estimators=280, learning_rate=0.020, max_depth=2, min_samples_leaf=5, subsample=0.82, random_state=SEED+104)
    ), FOLDS))
    specs.append(('hgb_risk_low', 'low', make_pipeline(
        SimpleImputer(strategy='median'),
        HistGradientBoostingRegressor(max_iter=300, learning_rate=0.025, max_leaf_nodes=8, l2_regularization=0.15, min_samples_leaf=8, random_state=SEED+105)
    ), FOLDS))
    specs.append(('ridge_risk_low', 'low', make_pipeline(
        SimpleImputer(strategy='median'), RobustScaler(),
        BayesianRidge()
    ), FOLDS))
    if XGB_OK:
        specs.append(('xgb_risk_low', 'low', XGBRegressor(
            n_estimators=620, max_depth=2, learning_rate=0.018, subsample=0.84, colsample_bytree=0.84,
            min_child_weight=2.0, reg_lambda=9.0, reg_alpha=0.10, objective='reg:squarederror',
            random_state=SEED+106, n_jobs=2, verbosity=0
        ), FOLDS))
    if LGB_OK:
        specs.append(('lgb_risk_phys', 'phys', LGBMRegressor(
            n_estimators=700, learning_rate=0.013, num_leaves=7, max_depth=3, min_child_samples=9,
            subsample=0.80, colsample_bytree=0.80, reg_alpha=0.15, reg_lambda=7.0,
            random_state=SEED+107, n_jobs=2, verbose=-1
        ), FOLDS))
    if CAT_OK:
        specs.append(('cat_risk_low', 'low', CatBoostRegressor(
            iterations=620, depth=2, learning_rate=0.018, l2_leaf_reg=12.0, loss_function='RMSE',
            random_seed=SEED+108, verbose=False, allow_writing_files=False,
            bootstrap_type='Bayesian', bagging_temperature=0.9, random_strength=1.1
        ), FOLDS))
    return specs

def cv_regression_risk_candidate(spec_name, fs_name, model, folds, target_kind='rank'):
    cols = FEATURE_SETS[fs_name]
    Xdf, Tdf = get_matrix(cols)
    if target_kind == 'rank':
        y_risk = event * (1.0 - np.log1p(np.minimum(time, 72.0)) / np.log1p(72.0))
    elif target_kind == 'inverse_time':
        y_risk = event / np.sqrt(np.maximum(time, 1.0))
    elif target_kind == 'multi_horizon':
        y_risk = 0.25 * binary_label_for_horizon(time, event, 12) + 0.35 * binary_label_for_horizon(time, event, 24) + 0.40 * binary_label_for_horizon(time, event, 48)
    else:
        y_risk = event.astype(float)
    oof = np.zeros(n_train, dtype=float)
    pred_test_folds = []
    counts = np.zeros(n_train, dtype=int)
    for tr_idx, va_idx in folds:
        Xtr = Xdf.iloc[tr_idx]
        Xva = Xdf.iloc[va_idx]
        ytr = y_risk[tr_idx]
        sw = (0.80 + 0.40 * TEST_DENSITY_W[tr_idx]).astype(float)
        sw *= np.where(event[tr_idx] == 1, 1.25, 0.85)
        sw = sw / np.mean(sw)
        try:
            pv, pt = reg_fit_predict(model, Xtr, ytr, Xva, Tdf, sample_weight=sw)
        except Exception:
            pv = np.full(len(va_idx), np.mean(ytr), dtype=float)
            pt = np.full(n_test, np.mean(ytr), dtype=float)
        oof[va_idx] += pv
        counts[va_idx] += 1
        pred_test_folds.append(pt)
    oof /= np.maximum(counts, 1)
    test_risk = np.mean(pred_test_folds, axis=0)
    return calibrate_risk_to_probs(oof, test_risk, spec_name)

for spec_name, fs_name, model, folds in build_regressor_specs():
    for target_kind in ['rank', 'inverse_time', 'multi_horizon']:
        try:
            p, q = cv_regression_risk_candidate(spec_name + '_' + target_kind, fs_name, model, FAST_FOLDS if 'ridge' in spec_name else folds, target_kind)
            add_candidate(f'reg_{spec_name}_{target_kind}', p, q, do_calibrated=True)
            rep = score_report(oof_candidates[-1], candidate_names[-1])
            print('REG', candidate_names[-1], round(rep['hybrid'], 6), round(rep['wbrier'], 6), round(rep['cindex'], 6))
        except Exception as ex:
            print('REG skipped:', spec_name, target_kind, ex)
            gc.collect()

# -------------------------- optional neural multi-horizon model --------------------------
def make_torch_candidates():
    if not TORCH_OK:
        return []
    torch.manual_seed(SEED)
    cols = FEATURE_SETS['low'] + [c for c in FEATURE_SETS['phys'] if c not in FEATURE_SETS['low']][:45]
    Xdf, Tdf = get_matrix(cols)
    imp = SimpleImputer(strategy='median')
    scaler = RobustScaler()
    Xmat = scaler.fit_transform(imp.fit_transform(Xdf))
    Tmat = scaler.transform(imp.transform(Tdf))
    Xmat = np.nan_to_num(Xmat, nan=0.0, posinf=0.0, neginf=0.0).astype('float32')
    Tmat = np.nan_to_num(Tmat, nan=0.0, posinf=0.0, neginf=0.0).astype('float32')
    Y = np.vstack([binary_label_for_horizon(time, event, H) for H in HORIZONS]).T.astype('float32')
    M = np.vstack([known_mask_for_horizon(time, event, H) for H in HORIZONS]).T.astype('float32')

    class Net(nn.Module):
        def __init__(self, d, hidden=48, dropout=0.10):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(d, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
                nn.Linear(hidden, hidden//2), nn.SiLU(), nn.Dropout(dropout),
                nn.Linear(hidden//2, 3)
            )
        def forward(self, x):
            logits = self.net(x)
            p = torch.sigmoid(logits)
            # soft monotonic penalty in loss; inference will enforce hard monotonicity.
            return p

    out = []
    for hidden, dropout, lr, epochs, name in [(40,0.08,0.012,520,'small'), (64,0.12,0.009,620,'med')]:
        oof = np.zeros((n_train, 4), dtype=float)
        counts = np.zeros(n_train, dtype=int)
        test_preds = []
        for fold_i, (tr_idx, va_idx) in enumerate(FAST_FOLDS):
            seed = SEED + 3000 + fold_i + hidden
            torch.manual_seed(seed)
            model = Net(Xmat.shape[1], hidden=hidden, dropout=dropout)
            opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.015)
            Xtr = torch.tensor(Xmat[tr_idx])
            Ytr = torch.tensor(Y[tr_idx])
            Mtr = torch.tensor(M[tr_idx])
            # density/event weights as row multipliers.
            rw = torch.tensor((0.80 + 0.35 * TEST_DENSITY_W[tr_idx]) * np.where(event[tr_idx]==1, 1.15, 0.95), dtype=torch.float32).view(-1, 1)
            model.train()
            for ep in range(epochs):
                opt.zero_grad()
                p = model(Xtr)
                bce = F.binary_cross_entropy(p.clamp(1e-5,1-1e-5), Ytr, reduction='none')
                loss = (bce * Mtr * rw).sum() / (Mtr * rw).sum().clamp_min(1.0)
                mono = F.relu(p[:,0] - p[:,1]).mean() + F.relu(p[:,1] - p[:,2]).mean()
                loss = loss + 0.08 * mono
                loss.backward()
                opt.step()
            model.eval()
            with torch.no_grad():
                pv = model(torch.tensor(Xmat[va_idx])).numpy()
                pt = model(torch.tensor(Tmat)).numpy()
            oof[va_idx, :3] += pv
            oof[va_idx, 3] += 1.0
            counts[va_idx] += 1
            test_preds.append(pt)
        oof = oof / np.maximum(counts[:, None], 1)
        q3 = np.mean(test_preds, axis=0)
        q = np.zeros((n_test, 4), dtype=float); q[:, :3] = q3; q[:, 3] = 1.0
        out.append((f'torch_{name}', enforce_monotonic(oof), enforce_monotonic(q)))
    return out

try:
    for name, p, q in make_torch_candidates():
        add_candidate(name, p, q, do_calibrated=True)
        rep = score_report(oof_candidates[-1], candidate_names[-1])
        print('TORCH', candidate_names[-1], round(rep['hybrid'], 6), round(rep['wbrier'], 6), round(rep['cindex'], 6))
except Exception as ex:
    print('Torch skipped:', ex)

print('Total raw candidates:', len(candidate_names))

# -------------------------- candidate pruning and robust blending --------------------------
def mad3(A, B):
    return float(np.mean(np.abs(A[:, :3] - B[:, :3])))

def select_candidates(max_keep=36):
    scores = []
    for name, P in zip(candidate_names, oof_candidates):
        rep = score_report(P, name)
        scores.append(rep['hybrid'])
    order = np.argsort(scores)[::-1]
    keep = []
    # Always keep a strong anchor and then greedily add diverse high OOF scorers.
    for idx in order:
        P = oof_candidates[idx]
        if len(keep) == 0:
            keep.append(idx)
            continue
        min_mad = min(mad3(P, oof_candidates[k]) for k in keep)
        # Dynamic diversity threshold: allows near-duplicates only if very highly scored.
        threshold = 0.0015 if len(keep) < 10 else 0.0030
        if min_mad > threshold or scores[idx] >= scores[keep[0]] - 0.0018:
            keep.append(idx)
        if len(keep) >= max_keep:
            break
    # Add a few smooth calibrated candidates even if clustered.
    for idx in order:
        if idx not in keep and ('prior' in candidate_names[idx] or 'phys' in candidate_names[idx] or 'logit' in candidate_names[idx] or 'reg_' in candidate_names[idx]):
            keep.append(idx)
        if len(keep) >= max_keep + 8:
            break
    keep = list(dict.fromkeys(keep))
    return keep, np.array(scores)

keep_idx, cand_scores = select_candidates(max_keep=34)
print('Kept candidates:', len(keep_idx))
for rank, idx in enumerate(keep_idx[:25], 1):
    print(f'{rank:02d}', candidate_names[idx], 'score=', round(float(cand_scores[idx]), 6), 'wb=', round(score_report(oof_candidates[idx])['wbrier'], 6))

O = np.stack([oof_candidates[i] for i in keep_idx], axis=0)
Q = np.stack([test_candidates[i] for i in keep_idx], axis=0)
K = O.shape[0]

def blend_with_weights(W, Ostack=O):
    W = np.asarray(W, dtype=float)
    W = np.clip(W, 0, None)
    if W.sum() <= 0:
        W = np.ones_like(W) / len(W)
    else:
        W = W / W.sum()
    return enforce_monotonic(np.tensordot(W, Ostack, axes=(0, 0)))

def objective_w(W):
    return -hybrid_score(blend_with_weights(W))

# Start from score-softmax weights.
s = cand_scores[keep_idx]
temp = max(0.0018, float(np.std(s)) * 0.70)
w0 = np.exp((s - s.max()) / temp)
w0 = w0 / w0.sum()

best_w = w0.copy()
best_score = hybrid_score(blend_with_weights(best_w))
print('Initial softmax blend:', round(best_score, 6))

# Greedy convex hill-climb: robust for small K and avoids extreme overfit from unconstrained optimization.
def greedy_hillclimb(start_w, rounds=70):
    w = np.asarray(start_w, dtype=float).copy()
    w = np.clip(w, 0, None); w = w / max(w.sum(), 1e-12)
    Pcur = blend_with_weights(w)
    scur = hybrid_score(Pcur)
    alphas = [0.015, 0.025, 0.04, 0.06, 0.085, 0.12, 0.17, 0.24, 0.32, 0.42]
    for _ in range(rounds):
        improved = False
        best_local = (scur, w)
        # Try each candidate as a direction.
        for i in range(K):
            ei = np.zeros(K); ei[i] = 1.0
            for a in alphas:
                ww = (1-a) * w + a * ei
                P = blend_with_weights(ww)
                sc = hybrid_score(P)
                # tiny complexity penalty for too many active weights
                active_pen = 0.000015 * np.sum(ww > 0.015)
                scp = sc - active_pen
                if scp > best_local[0] + 1e-7:
                    best_local = (scp, ww)
        if best_local[0] > scur + 1e-7:
            scur, w = best_local
            improved = True
        if not improved:
            break
    return w / w.sum(), hybrid_score(blend_with_weights(w))

starts = [w0, np.ones(K)/K]
for topn in [3,5,8,12,18]:
    ww = np.zeros(K)
    top = np.argsort(s)[::-1][:min(topn, K)]
    ww[top] = 1.0 / len(top)
    starts.append(ww)

for st in starts:
    ww, sc = greedy_hillclimb(st, rounds=55)
    if sc > best_score:
        best_score, best_w = sc, ww
        print('Greedy improved:', round(best_score, 6), 'active', int(np.sum(best_w > 0.005)))

# Optional SLSQP fine-tuning on the active set only.
active = np.where(best_w > 0.003)[0]
if SCIPY_OK and len(active) >= 2 and len(active) <= 18:
    def obj_active(x):
        ww = np.zeros(K)
        xx = np.clip(x, 0, None)
        xx = xx / max(xx.sum(), 1e-12)
        ww[active] = xx
        return -hybrid_score(blend_with_weights(ww))
    x0 = best_w[active] / best_w[active].sum()
    cons = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1.0},)
    bounds = [(0.0, 0.55) for _ in active]
    try:
        res = minimize(obj_active, x0, method='SLSQP', bounds=bounds, constraints=cons,
                       options={'maxiter': 600, 'ftol': 1e-10, 'disp': False})
        if res.success:
            ww = np.zeros(K)
            xx = np.clip(res.x, 0, None); xx = xx / max(xx.sum(), 1e-12)
            ww[active] = xx
            sc = hybrid_score(blend_with_weights(ww))
            if sc > best_score:
                best_score, best_w = sc, ww
                print('SLSQP improved:', round(best_score, 6), 'active', int(np.sum(best_w > 0.005)))
    except Exception as ex:
        print('SLSQP skipped:', ex)

# Random Dirichlet perturbations around the best/top candidates.
rng = np.random.default_rng(SEED + 909)
for it in range(1800):
    if it % 3 == 0:
        concentration = 80 * best_w + 0.10
    else:
        top = np.argsort(s)[::-1][:min(14, K)]
        concentration = np.full(K, 0.03)
        concentration[top] = np.exp((s[top]-s[top].max())/max(0.0018, temp)) * 2.0 + 0.08
    ww = rng.dirichlet(concentration)
    sc = hybrid_score(blend_with_weights(ww))
    # Favor sparse-ish blends when train score is tied.
    sc_adj = sc - 0.000010 * np.sum(ww > 0.02)
    if sc_adj > best_score + 1e-7:
        best_score, best_w = sc, ww
        print('Random improved:', it, round(best_score, 6), 'active', int(np.sum(best_w > 0.01)))

blend_oof = blend_with_weights(best_w)
blend_test = enforce_monotonic(np.tensordot(best_w / best_w.sum(), Q, axes=(0, 0)))
print('Best blend before final calibration:', score_report(blend_oof, 'blend'))

# -------------------------- final calibration, ranking-preserving shrinkage --------------------------
def final_calibrate(Poof, Ptest):
    Poof = enforce_monotonic(Poof)
    Ptest = enforce_monotonic(Ptest)
    bestP = Poof.copy()
    bestQ = Ptest.copy()
    base_score = hybrid_score(bestP)
    # Horizon-wise conservative calibration selected by OOF hybrid, not only Brier.
    for j, H in enumerate(HORIZONS):
        cand_pairs = []
        raw_o = bestP[:, j].copy(); raw_t = bestQ[:, j].copy()
        co, ct = _calibrate_1d(raw_o, raw_t, H, blend_grid=(0.05,0.12,0.20,0.32,0.45,0.60))
        for lam in [0.10, 0.18, 0.25, 0.35, 0.50, 0.70]:
            cand_pairs.append(((1-lam)*raw_o + lam*co, (1-lam)*raw_t + lam*ct, f'cal{lam}'))
        # Prior shrink: excellent Brier insurance on tiny samples.
        mask = known_mask_for_horizon(time, event, H)
        y = binary_label_for_horizon(time, event, H)
        prior = (y[mask].sum()+0.7)/(mask.sum()+1.4) if mask.sum() else np.mean(raw_o)
        for lam in [0.03, 0.06, 0.10, 0.14]:
            cand_pairs.append(((1-lam)*raw_o + lam*prior, (1-lam)*raw_t + lam*prior, f'prior{lam}'))
        # Temperature on logits.
        for temp_ in [0.82, 0.92, 1.08, 1.20, 1.38]:
            center = np.mean(logit(np.clip(raw_o[mask],1e-5,1-1e-5))) if mask.sum() else 0.0
            oo = expit(center + (logit(raw_o) - center) / temp_)
            tt = expit(center + (logit(raw_t) - center) / temp_)
            cand_pairs.append((oo, tt, f'temp{temp_}'))
        local_best = (base_score, raw_o, raw_t, 'raw')
        for oo, tt, label in cand_pairs:
            Ptry = bestP.copy(); Qtry = bestQ.copy()
            Ptry[:, j] = oo; Qtry[:, j] = tt
            Ptry = enforce_monotonic(Ptry); Qtry = enforce_monotonic(Qtry)
            sc = hybrid_score(Ptry)
            if sc > local_best[0] + 2e-7:
                local_best = (sc, Ptry[:, j].copy(), Qtry[:, j].copy(), label)
        if local_best[0] > base_score + 2e-7:
            bestP[:, j] = local_best[1]
            bestQ[:, j] = local_best[2]
            bestP = enforce_monotonic(bestP); bestQ = enforce_monotonic(bestQ)
            base_score = local_best[0]
            print('Final cal improved H', H, local_best[3], round(base_score, 6))
    return enforce_monotonic(bestP), enforce_monotonic(bestQ)

final_oof, final_test = final_calibrate(blend_oof, blend_test)

# p12 tuning: only affects ranking/monotonicity, not the official weighted Brier horizons.
def tune_p12_for_rank(Poof, Ptest):
    Poof = enforce_monotonic(Poof); Ptest = enforce_monotonic(Ptest)
    base = hybrid_score(Poof)
    best = (base, Poof[:,0].copy(), Ptest[:,0].copy(), 'raw')
    p12 = Poof[:,0]; q12 = Ptest[:,0]
    p24 = Poof[:,1]; q24 = Ptest[:,1]
    # Risk-regressor consensus as alternative early-ranking signal.
    reg_idxs = [i for i, nm in enumerate(candidate_names) if nm.startswith('reg_')]
    if reg_idxs:
        # Use top few reg candidates by OOF score.
        reg_idxs = sorted(reg_idxs, key=lambda i: cand_scores[i] if i < len(cand_scores) else hybrid_score(oof_candidates[i]), reverse=True)[:8]
        R_o = np.mean([oof_candidates[i][:,0] for i in reg_idxs], axis=0)
        R_t = np.mean([test_candidates[i][:,0] for i in reg_idxs], axis=0)
    else:
        R_o, R_t = p12, q12
    variants = []
    for a in [0.10,0.20,0.35,0.50,0.65]:
        variants.append(((1-a)*p12 + a*np.minimum(p24, R_o), (1-a)*q12 + a*np.minimum(q24, R_t), f'reg{a}'))
        variants.append(((1-a)*p12 + a*(0.42*p24), (1-a)*q12 + a*(0.42*q24), f'scale24_{a}'))
    for oo, tt, lab in variants:
        Ptry = Poof.copy(); Qtry = Ptest.copy()
        Ptry[:,0] = np.minimum(oo, Ptry[:,1])
        Qtry[:,0] = np.minimum(tt, Qtry[:,1])
        Ptry = enforce_monotonic(Ptry); Qtry = enforce_monotonic(Qtry)
        sc = hybrid_score(Ptry)
        if sc > best[0] + 2e-7:
            best = (sc, Ptry[:,0].copy(), Qtry[:,0].copy(), lab)
    if best[0] > base + 2e-7:
        Poof[:,0] = best[1]; Ptest[:,0] = best[2]
        print('p12 rank tune:', best[3], round(best[0], 6))
    return enforce_monotonic(Poof), enforce_monotonic(Ptest)

final_oof, final_test = tune_p12_for_rank(final_oof, final_test)
print('Final OOF report:', score_report(final_oof, 'final'))

# -------------------------- strict submission validation --------------------------
submission = pd.DataFrame({ID_COL: sample_sub[ID_COL].values})
for j, c in enumerate(PROB_COLS):
    submission[c] = final_test[:, j]
submission = submission[REQUIRED_COLS]

# Hard validation against contest rules.
assert list(submission.columns) == REQUIRED_COLS
assert submission[ID_COL].astype(str).tolist() == sample_sub[ID_COL].astype(str).tolist()
assert submission[ID_COL].is_unique
vals = submission[PROB_COLS].to_numpy(dtype=float)
assert np.all(np.isfinite(vals))
assert np.all((vals >= -1e-12) & (vals <= 1 + 1e-12))
assert np.all(vals[:,0] <= vals[:,1] + 1e-12)
assert np.all(vals[:,1] <= vals[:,2] + 1e-12)
assert np.all(vals[:,2] <= vals[:,3] + 1e-12)

# Ensure exact range and exact monotonicity after DataFrame assignment.
vals = enforce_monotonic(vals)
submission.loc[:, PROB_COLS] = vals

out_path = '/kaggle/working/submission.csv' if os.path.isdir('/kaggle/working') else 'submission.csv'
submission.to_csv(out_path, index=False)
submission.to_csv('submission.csv', index=False)

# Diagnostics only; Kaggle uses submission.csv.
summary = {
    'input_dir': os.path.dirname(train_path),
    'n_candidates_total': int(len(candidate_names)),
    'n_candidates_kept': int(len(keep_idx)),
    'best_oof_report': score_report(final_oof, 'final'),
    'active_blend': [
        {'name': candidate_names[keep_idx[i]], 'weight': float(best_w[i]), 'oof_score': float(cand_scores[keep_idx[i]])}
        for i in np.argsort(-best_w) if best_w[i] > 0.003
    ]
}
with open('blend_diagnostics.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved:', out_path)
print(submission.head())
print('Active blend:')
for item in summary['active_blend'][:30]:
    print(f"{item['weight']:.4f} | {item['oof_score']:.6f} | {item['name']}")


Input directory: /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
Train/test: (221, 37) (95, 35)
Events/censored: 69 152
Feature counts: {'all': 168, 'orig': 34, 'phys': 146, 'low': 26}
Test-density weights: 0.45 2.021571565491674 1.004662890581401
KNN candidate: knn_low_robust_k7_p1.2_cal 0.9269035714669686
KNN candidate: knn_low_robust_k13_p1.0_cal 0.923421226765559
KNN candidate: knn_low_standard_k19_p1.0_cal 0.959740018779488
KNN candidate: knn_phys_robust_k11_p1.0_cal 0.9271986808337176
KNN candidate: knn_phys_robust_k23_p0.8_cal 0.9418796750781109
KNN candidate: knn_phys_quantile_k17_p1.0_cal 0.9595637376890438
KNN candidate: knn_all_robust_k15_p0.8_cal 0.9351123301083433
KNN candidate: knn_all_quantile_k25_p0.7_cal 0.9587951172965504
KNN candidate: knn_orig_robust_k11_p1.1_cal 0.8782741500626028
Classifier specs: 21 xgb/lgb/cat: True True True
BIN bin_et_leaf2_all_cal 0.970097 0.015995 0.937645
BIN bin_et_leaf4_all_cal 0.971527 0.014733 0.939467
BIN bin_et_leaf7_all_cal 0